# FOLTRA Week 6: Field-Style Robust Augmentation

This workflow clones a pinned repository commit, verifies a GPU and both datasets, runs tests and a smoke run, then (only after an explicit flag) trains the 10-epoch candidate and evaluates its frozen checkpoint on PlantVillage and PlantDoc. PlantDoc is present only as a test-only archive and is never passed to training.

Before opening Colab, run `python -B -m scripts.colab.package_week_06` locally and upload the three generated ZIPs from `experiments/week_06_robust_augmentation/training/outputs/colab_upload/` to `MyDrive/FOLTRA/week_06/`. Commit and push the implementation, then put that exact commit SHA below.


In [ ]:
REPOSITORY_URL = 'https://github.com/Akshith-cdr/Foltra.git'
EXPECTED_COMMIT = 'SET_TO_EXACT_PUSHED_COMMIT_SHA'
RUN_FULL_TRAINING = False
if EXPECTED_COMMIT.startswith('SET_'):
    raise ValueError('Set EXPECTED_COMMIT to the exact pushed commit SHA before running.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv, json, os, shutil, subprocess, sys, uuid, zipfile
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0))
DRIVE_INPUT = Path('/content/drive/MyDrive/FOLTRA/week_06')
for name in ('plantvillage_color.zip', 'plantdoc_test.zip', 'baseline_reference.zip'):
    if not (DRIVE_INPUT / name).is_file():
        raise FileNotFoundError(DRIVE_INPUT / name)
RUN_ROOT = DRIVE_INPUT / 'runs' / ('run_' + uuid.uuid4().hex[:8])
RUN_ROOT.mkdir(parents=True)
PROJECT = RUN_ROOT / 'Foltra'
subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--detach', EXPECTED_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if actual != EXPECTED_COMMIT:
    raise RuntimeError(f'Commit mismatch: {actual} != {EXPECTED_COMMIT}')
print('Verified repository commit:', actual)


In [ ]:
LOCAL_DATA = Path('/content/foltra_week06_data_' + uuid.uuid4().hex[:8])
LOCAL_DATA.mkdir()
for archive_name in ('plantvillage_color.zip', 'plantdoc_test.zip'):
    local_archive = LOCAL_DATA / archive_name
    shutil.copy2(DRIVE_INPUT / archive_name, local_archive)
    with zipfile.ZipFile(local_archive) as archive:
        archive.extractall(LOCAL_DATA)
    local_archive.unlink()
with zipfile.ZipFile(DRIVE_INPUT / 'baseline_reference.zip') as archive:
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)
import yaml
dataset_config_path = PROJECT / 'configs/datasets.yaml'
dataset_config = yaml.safe_load(dataset_config_path.read_text())
dataset_config['paths']['dataset_root'] = str(LOCAL_DATA / 'Datasets')
dataset_config_path.write_text(yaml.safe_dump(dataset_config, sort_keys=False))


In [ ]:
training_config = yaml.safe_load((PROJECT / 'configs/robust_augmentation.yaml').read_text())
for split, manifest_name in training_config['manifests'].items():
    with (PROJECT / 'data/manifests' / manifest_name).open(newline='', encoding='utf-8-sig') as handle:
        rows = list(csv.DictReader(handle))
    missing = [row['path'] for row in rows if not (LOCAL_DATA / row['path']).is_file()]
    if missing:
        raise FileNotFoundError(f'{split}: {len(missing)} missing; first: {missing[0]}')
    print(split, len(rows), 'PlantVillage files verified')
for required in ('meta.json', 'test/img', 'test/ann'):
    path = LOCAL_DATA / 'Datasets/plantdoc' / required
    if not path.exists():
        raise FileNotFoundError(path)
print('PlantDoc test-only inputs verified')
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


Run the bounded smoke training first. It uses the same ImageNet initialization and augmentation code but only eight samples per split for one epoch.


In [ ]:
subprocess.run([sys.executable, '-u', '-B', '-m', 'src.training.train_robust_augmentation', '--smoke'], check=True)


Set `RUN_FULL_TRAINING = True` in the parameter cell and rerun this cell to start the controlled 10-epoch GPU experiment. A new output directory is created, so prior runs are not overwritten.


In [ ]:
if not RUN_FULL_TRAINING:
    print('Full training not started. Set RUN_FULL_TRAINING = True when ready.')
else:
    before = set((PROJECT / 'experiments/week_06_robust_augmentation/training/outputs').glob('train_*'))
    subprocess.run([sys.executable, '-u', '-B', '-m', 'src.training.train_robust_augmentation'], check=True)
    after = set((PROJECT / 'experiments/week_06_robust_augmentation/training/outputs').glob('train_*'))
    created = sorted(after - before)
    if len(created) != 1 or not (created[0] / 'best.pt').is_file():
        raise RuntimeError(f'Expected exactly one completed training run, observed: {created}')
    checkpoint = (created[0] / 'best.pt').relative_to(PROJECT).as_posix()
    subprocess.run([sys.executable, '-u', '-B', '-m', 'src.evaluation.evaluate_robust_augmentation', '--checkpoint', checkpoint, '--device', 'cuda'], check=True)
    print('Training:', created[0])
    print('Evaluation root:', PROJECT / 'experiments/week_06_robust_augmentation/evaluation/outputs')
